In [0]:
# 1. Read stream from the Bronze table
bronze_df = (
    spark.readStream
    .table("realtime_fraud_detection.01_bronze.raw_transactions")
)

# 2. Perform any light cleaning/casting if needed (e.g., ensuring columns are clean)
silver_df = bronze_df.dropDuplicates(["Time", "Amount"]) # Optional safety check for duplicate rows

# 3. Write the cleaned data into the Silver Delta table using Auto Loader / Delta streaming
query = (
    silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/realtime_fraud_detection/00_landing/raw_data/transactions/_checkpoints/silver")
    .table("realtime_fraud_detection.02_silver.cleaned_transactions")
)

print("🥈 Silver Layer Stream Ingestion Completed Successfully!")